# 10 · MVP demo: Skin Lesion Triage

**What this notebook does:** it starts the product as a web app with a public address, for an investor or class demo.
A patient sends a photo and six answers → the image model and the questionnaire model score them → the patient is
ranked in today's queue → the **live agent** (Gemini) prepares the case → the dermatologist gets the morning list and a Word report.

**Where the code comes from:**
1. The repository is cloned fresh from GitHub (`BRANCH` in Cell 1).
2. The MVP package `skin_triage_mvp.zip` is then taken from the **shared project folder on Drive** (`PROJECT/mvp/`)
   and laid over the clone: every file in the zip replaces the file with the same path. This is how a change to the agent
   or the app reaches the demo before it is merged on GitHub. Once it is merged, the zip is no longer needed.

**Before running (once):**
- Put the MVP files in `PROJECT/mvp/` on Drive: the folders `app/` and `agent/` (extracted), or `skin_triage_mvp.zip`.
- Colab Secrets (key icon, left bar): `GEMINI_API_KEY`, with **Notebook access** switched on.
- Optional: Runtime → Change runtime type → **T4 GPU** (the image model is faster; CPU also works).

Then: **Runtime → Run all**. The last cell prints the app's address.

In [1]:
# Cell 1 · Configuration (same project folder as every notebook)
import os, sys, subprocess, shutil, zipfile, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT = "/content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers"
else:
    PROJECT = os.path.abspath("..")

MVP_ZIP  = os.path.join(PROJECT, "mvp", "skin_triage_mvp.zip")   # the MVP package, shared on Drive
REPO_URL = "https://github.com/IfatDav/skin-lesion-assistant.git"
BRANCH   = "main"                     # e.g. "audit-fixes" to demo a branch before it is merged
WORK     = "/content/skin_triage_app" # a fresh working copy; nothing here is written back to Drive or GitHub
PORT     = 8501

print("project folder:", PROJECT, "| exists:", os.path.isdir(PROJECT))
print("MVP package:   ", MVP_ZIP, "| exists:", os.path.isfile(MVP_ZIP))

Mounted at /content/drive
project folder: /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers | exists: True
MVP package:    /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/mvp/skin_triage_mvp.zip | exists: False


In [10]:
# Cell 2 · Code: fresh clone from GitHub, then the MVP files from Drive laid over it
import glob
os.chdir("/content" if IN_COLAB else PROJECT)   # step out first: a rerun deletes WORK
if os.path.isdir(WORK):
    shutil.rmtree(WORK)
subprocess.run(["git", "clone", "--quiet", "--depth", "1", "-b", BRANCH, REPO_URL, WORK], check=True)
print(f"cloned {BRANCH} → {WORK}")

# The MVP files: an extracted folder in PROJECT/mvp/ (app/, agent/, ...) or skin_triage_mvp.zip there
MVP_DIR = os.path.join(PROJECT, "mvp")
hits = glob.glob(os.path.join(MVP_DIR, "**", "app", "streamlit_app.py"), recursive=True)
if hits:
    src_root = os.path.dirname(os.path.dirname(hits[0]))
    replaced = []
    for path in glob.glob(os.path.join(src_root, "**", "*"), recursive=True):
        if os.path.isfile(path) and not path.endswith(".zip"):
            rel = os.path.relpath(path, src_root)
            dst = os.path.join(WORK, rel)
            if os.path.exists(dst):
                replaced.append(rel)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(path, dst)
    print("MVP files copied from:", src_root)
    for r in replaced:
        print("   replaced:", r)
elif os.path.isfile(MVP_ZIP):
    with zipfile.ZipFile(MVP_ZIP) as z:
        z.extractall(WORK)
    print("MVP package unzipped from:", MVP_ZIP)
else:
    print("No MVP files in", MVP_DIR, "| contents:", os.listdir(MVP_DIR) if os.path.isdir(MVP_DIR) else "folder missing")
    print("Using the GitHub code only.")

assert os.path.isfile(os.path.join(WORK, "app", "streamlit_app.py")), (
    "app/ is missing: put the MVP files in PROJECT/mvp/ (note: 'mvp', lowercase), or set BRANCH to a branch that has app/.")
os.chdir(WORK)

cloned main → /content/skin_triage_app
MVP files copied from: /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/mvp
   replaced: agent/triage_agent.py


In [3]:
# Cell 3 · Install (PyTorch is already in Colab; this adds the rest)
!pip install -q streamlit timm python-docx pydantic "scikit-learn==1.9.1" crewai google-genai litellm chromadb sentence-transformers
print("installed")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
installed


In [11]:
# Cell 4 · The Gemini key: from Colab Secrets, or typed once if Secrets are not reachable
from getpass import getpass
key = None
try:
    from google.colab import userdata
    key = userdata.get("GEMINI_API_KEY")
except Exception as e:   # not added, Notebook access off, or the secret request timed out
    print(f"Colab Secrets did not return the key ({type(e).__name__}).")
    print("Fix it in the key icon on the left (Notebook access ON), or paste the key now:")
if not key:
    key = getpass("GEMINI_API_KEY: ")
os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"] = key
os.environ["RAG_DIR"] = os.path.join(WORK, "data", "processed", "rag")
os.environ["CREWAI_TRACING_ENABLED"] = "false"     # no "Share this execution trace?" question (20 s wait)
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
# os.environ["TRIAGE_MODEL"] = "gemini-3.5-flash"   # optional: a different Gemini model
print("key loaded (not printed)")

key loaded (not printed)


In [12]:
# Cell 5 · The guidance index the agent cites from (NCI PDQ, 724 passages; about a minute)
!python agent/rag.py build

Loading weights: 100% 103/103 [00:00<00:00, 4326.97it/s]
Batches: 100% 23/23 [00:01<00:00, 12.70it/s]
indexed 724 chunks from 7 documents


In [6]:
# Cell 6 · The image model: downloads the weights once and checks they load
!python -c "from src.predict import VisualRiskModel as V; m=V(); print('ConvNeXt ready on', m.device, '| test score:', m.predict(['app/examples/nevus.jpg'])['cancer_score'])"

downloading convnext_pad_s42_scc2.safetensors from https://github.com/IfatDav/skin-lesion-assistant/releases/download/image-model-v1/
ConvNeXt ready on cuda | test score: 0.115


In [13]:
# Cell 7 · Rehearsal: one live agent call, so a problem shows here and not in front of the investor
!python -c "import sys; sys.path.insert(0,'app'); import engine as E; from PIL import Image; ans=dict(itch='yes',grew='yes',hurt='no',changed='no',bleed='no',elevation='yes'); r=E.rank_today(0.97, E.questionnaire_score(66,'NECK',ans)); c=E.make_case('NEW-01',r['rank'],r,66,'NECK',ans,'השומה החשיכה בחודש האחרון','app/examples/contradiction.jpg',E.photo_quality(Image.open('app/examples/contradiction.jpg'))); rv,how=E.review(c,'live'); print('engine:', how); print('decision:', rv.decision); print('message:', rv.follow_up_question or rv.retake_instruction)"
# Expected: engine: live agent · decision: ask_patient · a question in Hebrew.
# If it says "live agent unavailable", read the reason in brackets.

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
╭────────────────────────────── 🤖 Agent Started ──────────────────────────────╮
│                                                                              │
│  Agent: Clinician Seat triage evidence assistant                             │
│                                                                              │
│  Task: Review patient NEW-01.                                                │
│                                                                              │
│  Steps:                                                                      │
│  1. Call get_case_data, predict_visual_risk, score_symptoms and              │
│  check_image_quality for this patient.                                       │
│  2. Call retrieve_guidance_context with a short query built from the         │
│  symptoms answered "yes" and the body site.      

In [14]:
# Cell 8 · Start the app and give it a public address
from IPython.display import HTML, display
subprocess.run("pkill -f 'streamlit run' ; pkill -f cloudflared", shell=True)
app_log = open("/content/streamlit.log", "w")
subprocess.Popen(["streamlit", "run", "app/streamlit_app.py", "--server.port", str(PORT), "--server.headless", "true",
                  "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],   # uploads work through the tunnel
                 cwd=WORK, env=os.environ.copy(), stdout=app_log, stderr=subprocess.STDOUT)
if not os.path.exists("/content/cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    "-O", "/content/cloudflared"], check=True)
    os.chmod("/content/cloudflared", 0o755)
tunnel_log = open("/content/tunnel.log", "w")
subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"], stdout=tunnel_log, stderr=subprocess.STDOUT)

url = None
for _ in range(30):
    time.sleep(2)
    for line in open("/content/tunnel.log"):
        if "trycloudflare.com" in line and "https://" in line:
            url = line[line.index("https://"):].split()[0]
    if url:
        break
if url:
    display(HTML(f'<p style="font-size:22px">The app is live: <a href="{url}" target="_blank">{url}</a></p>'
                 '<p>The address works while this notebook is running.</p>'))
else:
    print("No address yet. Run Cell 9 to see the logs.")

In [9]:
# Cell 9 · Troubleshooting: the last lines of each log
!tail -n 25 /content/streamlit.log
!tail -n 5 /content/tunnel.log



2026-11-09 07:06:16.765 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.220.251:8501

2026-11-09T07:06:18Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
2026-11-09T07:06:18Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026-11-09T07:06:18Z INF Starting metrics server on 127.0.0.1:20241/metrics
2026-11-09T07:06:18Z INF Tunnel connection curve preferences: [X25519MLKEM768 CurveID(65074) CurveP256] connIndex=0 event=0 ip=198.41.200.13
2026-11-09T07:06:18Z INF Registered tunnel connection connIndex=0 connection=aab73ddf-96e3-4d36-8355-87e99f2451ee event=0 ip=198.41.200.13 location=lax13 protocol=quic
